# Orchestrator

This notebook runs the datasource adapter to produce canonical data and submits it to the `CanonicalHandler` for storage.

To (re-)run an experiment, simply change `dataset_id` and `version` and re-run.

## 1. Inputs

In [1]:
dataset_id = "unsw_nb15"
version = "v1"

## 2. Setup

In [2]:
import sys
from pathlib import Path

# Ensure the project root is importable
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [3]:
from context_based_anomalous_flow_detector.data_processing.canonical_handler import CanonicalHandler
from context_based_anomalous_flow_detector.data_processing.adapters.unsw_nb15_adapter import create_canonical_data

## 3. Run

Instantiate a `CanonicalHandler` and submit the adapter function. The handler executes it and saves the canonical data to
`data/<dataset_id>/<version>/canonical/canonical.parquet`.

In [4]:
handler = CanonicalHandler()
target_dir = handler.create(create_canonical_data, dataset_id, version)

print("Canonical data written to:", target_dir)

Canonical data saved to: /home/irene/DataSSD/Code/context_based_anomalous_flow_detector/data/unsw_nb15/v1/canonical/canonical.parquet
Canonical data written to: /home/irene/DataSSD/Code/context_based_anomalous_flow_detector/data/unsw_nb15/v1/canonical


## 4. Inspect result

In [5]:
import pandas as pd

canonical = pd.read_parquet(target_dir / "canonical.parquet")
canonical.head()

,bytes_out,bytes_in,packets_out,packets_in,tcpflags,ttl,ipsrc,ipdst,portsrc,portdst,duration,starttime,proto
0,178,146,2,2,0,31,59.166.0.2,149.171.126.3,4894,53,2,1424242193040,17
1,2976,4704,28,28,27,32,59.166.0.4,149.171.126.6,52671,31992,335,1424242192744,6
2,548216,13662,438,238,27,32,59.166.0.0,149.171.126.9,47290,6881,2460,1424242190649,6
3,178,146,2,2,0,31,59.166.0.8,149.171.126.7,43310,53,1,1424242193145,17
4,162,130,2,2,0,31,59.166.0.1,149.171.126.1,45870,53,1,1424242193239,17


In [6]:
print(f"Rows: {len(canonical):,}")
print(f"Columns: {list(canonical.columns)}")

Rows: 2,365,424
Columns: ['bytes_out', 'bytes_in', 'packets_out', 'packets_in', 'tcpflags', 'ttl', 'ipsrc', 'ipdst', 'portsrc', 'portdst', 'duration', 'starttime', 'proto']


## 5. Preprocessing

The `PreprocessingHandler` reads the canonical data, validates it against the schema, and produces model-input tensors.
It records provenance: which canonical data (dataset_id/version) was consumed and the preprocessing configuration.

In [7]:
from context_based_anomalous_flow_detector.data_processing.preprocessing_handler import PreprocessingHandler
from context_based_anomalous_flow_detector.data_processing.preprocessing import PreprocessParams

In [8]:
preprocessor = PreprocessingHandler()
params = PreprocessParams(seq_len=34, group_cols=["ipsrc"], time_col="starttime")
preprocessed_dir = preprocessor.run(dataset_id, version, params=params)

print("Preprocessed data written to:", preprocessed_dir)

Preprocessed tensors saved to: /home/irene/DataSSD/Code/context_based_anomalous_flow_detector/data/unsw_nb15/v1/preprocessed/tensors.pt
Preprocessed data written to: /home/irene/DataSSD/Code/context_based_anomalous_flow_detector/data/unsw_nb15/v1/preprocessed


In [9]:
import torch

tensors = torch.load(preprocessed_dir / "tensors.pt", weights_only=False)
for name, split in tensors.items():
    print(name, {k: tuple(v.shape) for k, v in split.items()})

train {'cont': (48715, 34, 13), 'proto': (48715, 34), 'port': (48715, 34), 'mask': (48715, 34)}
eval {'cont': (10438, 34, 13), 'proto': (10438, 34), 'port': (10438, 34), 'mask': (10438, 34)}
test {'cont': (10437, 34, 13), 'proto': (10437, 34), 'port': (10437, 34), 'mask': (10437, 34)}


## 6. Training

The `TrainerHandler` reads the preprocessed tensors of a dataset/version, trains a `MiniBert` model on the train split,
and saves the artifact plus a provenance manifest to `models/<dataset_id>/<version>/`.

In [10]:
from context_based_anomalous_flow_detector.training.trainer_handler import TrainerHandler, TrainingParams

In [11]:
trainer = TrainerHandler()
params = TrainingParams(seed=42, batch_size=64, num_epochs=5, d_model=128, n_heads=4, num_layers=4)
model_dir = trainer.run(dataset_id, version, params=params)

print("Trained model written to:", model_dir)

Using hardware accelerator device: cuda
Epoch 1/5 complete. Mean batch loss: 6.979151
Epoch 2/5 complete. Mean batch loss: 5.346859
Epoch 3/5 complete. Mean batch loss: 4.971990
Epoch 4/5 complete. Mean batch loss: 3.703979
Epoch 5/5 complete. Mean batch loss: 3.182853
Trained model saved to: /home/irene/DataSSD/Code/context_based_anomalous_flow_detector/models/unsw_nb15/v1/model.pt
Trained model written to: /home/irene/DataSSD/Code/context_based_anomalous_flow_detector/models/unsw_nb15/v1


In [12]:
import yaml

from context_based_anomalous_flow_detector.data_processing._manifest import read_manifest

manifest = read_manifest(model_dir)
print(yaml.safe_dump(manifest, sort_keys=False))

dataset_id: unsw_nb15
version: v1
created_at: '2026-09-04T21:13:21.401886+00:00'
training_params:
  seed: 42
  batch_size: 64
  num_epochs: 5
  learning_rate: 0.0001
  weight_decay: 1.0e-05
  device: auto
  model:
    name: mini_bert
    num_cont_features: 13
    d_model: 128
    n_heads: 4
    num_layers: 4
input:
  dataset_id: unsw_nb15
  version: v1
  preprocessed_dir: /home/irene/DataSSD/Code/context_based_anomalous_flow_detector/data/unsw_nb15/v1/preprocessed
  preprocess_params:
    seq_len: 34
    group_cols:
    - ipsrc
    time_col: starttime
    split:
    - 0.7
    - 0.85
    log_div:
      bytes_in: 20.0
      bytes_out: 20.0
      packets_in: 12.0
      packets_out: 12.0
      duration: 12.0
      time_delta: 15.0
    ttl_max: 255.0
metrics:
  epoch_1_train_loss: 6.9791512050703854
  epoch_2_train_loss: 5.346858779641551
  epoch_3_train_loss: 4.9719897143630885
  epoch_4_train_loss: 3.703978886898808
  epoch_5_train_loss: 3.1828534211498365
output_file: model.pt



## 7. Evaluation

Training uses masked-token prediction: ~15% of the real tokens are masked and the model must reconstruct them.
The `EvaluationHandler` applies the same masking to the held-out split and measures how well the model recovers
the masked tokens (continuous MSE/MAE, proto/port accuracy), alongside trivial baselines for context.

In [13]:
from context_based_anomalous_flow_detector.evaluation.evaluation_handler import EvaluationHandler, EvalParams

In [14]:
evaluator = EvaluationHandler()
eval_params = EvalParams(seed=42, mask_ratio=0.15, batch_size=64, top_k=5)
eval_model_dir = evaluator.run(dataset_id, version, split="test", params=eval_params)

print("Evaluation results written to:", eval_model_dir / "evaluation.json")

Using hardware accelerator device: cuda


/home/irene/miniconda3/envs/context_git/lib/python3.14/site-packages/torch/nn/modules/transformer.py:529: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /pytorch/aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = torch._nested_tensor_from_mask(



Masked-token evaluation on split 'test' (unsw_nb15/v1):
  evaluated batches : 164
  masked positions  : 53,322
  model:
    cont MSE/MAE    : 0.016890 / 0.063970  (naive-zero MAE: 0.354348)
    proto top-1 acc : 0.9518  (majority: 0.7780, chance: 0.0039)
    port top-1 acc  : 0.522261  (majority: 0.213477, chance: 0.000015)
    port top-5 acc : 0.658509  (chance: 0.000076)
Evaluation results written to: /home/irene/DataSSD/Code/context_based_anomalous_flow_detector/models/unsw_nb15/v1/evaluation.json


Inspect the raw evaluation results (metrics + baselines) from the saved JSON.

In [15]:
import json

with open(eval_model_dir / "evaluation.json") as f:
    results = json.load(f)

print("metrics:", json.dumps(results["metrics"], indent=2))

metrics: {
  "evaluated_batches": 164,
  "masked_positions": 53322,
  "cont_mse": 0.016890259168201097,
  "cont_mae": 0.06396961384681536,
  "proto_top1_acc": 0.9518397659502644,
  "port_top1_acc": 0.5222609804583473,
  "port_top5_acc": 0.6585086830951578,
  "baselines": {
    "proto_majority_top1_acc": 0.7780278309140692,
    "port_majority_top1_acc": 0.21347661378042834,
    "proto_chance_top1": 0.0038910505836575876,
    "port_chance_top1": 1.5258556235409006e-05,
    "port_chance_top5": 7.629278117704503e-05,
    "cont_naive_zero_mae": 0.3543483307978965
  }
}
